# Introduction to Bayesian Optimisation (BO)

This notebook builds a minimal, visual BO loop for a 1D toy function. The aim is to see _how_ BO works, how it chooses new experiments, and _why_ it balances exploration vs exploitation.

We will:

- Define an unknown objective function (think: an expensive chemistry experiment).
- Fit a Gaussian Process (GP) surrogate model.
- Use Expected Improvement (EI) to pick the next experiment.
- Track best-so-far and regret.


## What is Bayesian Optimisation?

Bayesian Optimisation (BO) is a strategy for **finding the best settings of an experiment** when each experiment is expensive, slow, or noisy.

Key ideas:

- Treat the experiment as a **black box**: we can choose inputs and observe outputs, but we do not know the true function.
- Build a **surrogate model** that predicts outcomes _and uncertainty_ from the data collected so far.
- Use an **acquisition function** to decide the next experiment by trading off exploration (learn more) and exploitation (aim for high yield).

BO is widely used in chemistry because running experiments is costly and the design space is large.

Importantly, BO is not the same as model-making, but relies on models. This distinction is important to keep in mind.


## Model-making: why we build a model at all

Linear regression is one method for model building; as scienists use it to inter/extrapolate simple data, and it has predictive capability. BO also builds models, but with an added requirement: **BO needs uncertainty**, not just predictions.

Why?

- BO aims to **decide the next experiment**, rather than just explain trends.
- A model with uncertainty tells us _where we are confident_ and _where we should explore_.

In this notebook, the model is a **Gaussian Process**. It behaves like a regression model (think linear regression), but also provides uncertainty estimates that BO can use directly.


**Before you start:** run the whole notebook top-to-bottom so the figures are generated.


### Why Gaussian Processes?

A Gaussian Process (GP) is a **probabilistic model over functions**. Instead of a single prediction, it gives a distribution:

- **Mean** = best guess of the function value at each x.
- **Uncertainty (standard deviation)** = how unsure the model is there.

This uncertainty is crucial: BO needs it to decide _where to learn next_. Conceptually, GPs are like having a Gaussian distribution at every point on a model line such that they form a continuous uncertainty bound. You'll see this visually below.


### GP vs linear and higher-order regression

Traditionally, it is taught that linear regression produces a straight line which relates `f(x)` to `x`. It works well when the relationship is linear, but on its own, it cannot be used to express curves.

A common extension is **polynomial regression**, which adds powers of x (x^2, x^3, ...). \*note this is still linear regression, see Notebook 2 for more details

This can fit curves, but:

- You must choose the degree in advance.
- High-degree polynomials can overfit and behave wildly between data points.
- Uncertainty quantification is not built-in.

A GP can be viewed as a **flexible, non-parametric generalisation** of regression and interpolation:

- It adapts to data without choosing a fixed functional form.
- It represents a distribution over functions, not a single curve.
- It provides **uncertainty** that grows away from data, which is essential for BO. 

You can think of a GP as a form of _probabilistic interpolation_: it smoothly connects observed points while expressing how confident it is between them.

**Figure 1:** Linear, polynomial, and GP fits on the same data.

<img src="assets/gp_vs_linear.png" width="420" alt="Linear, polynomial, and GP fits" />

**Figure 2:** Samples from the GP posterior (possible functions consistent with data).

<img src="assets/gp_posterior_samples.png" width="420" alt="GP posterior samples" />

In the first figure, notice the key difference between regular regression and a GP: regression gives a single best-fit line, but a GP gives a _distribution_ over lines. That means the GP can say **how uncertain it is**, and that uncertainty changes when we add or remove data.

Note that in this case, in the small data regime, uncertainty is apparently low at the data points and counterintuitively it appears to get bigger with more data. In building the GP, we essentially ask "What is a smooth line that is consistent with my data?" When there isn't much data, the GP knows the line must go through the points (uncertainty collapses to zero), but uncertainty is far away from the points. When we have lots of data, the GP can learn about (apparent) noise in the system, which is reflected in the uncertainty bounds. This effect is not universal: if the data are highly consistent or the noise level is known to be small, increasing data typically reduces uncertainty throughout.

Think of this in the context of linear regression as the difference between drawing a perfectly-fitting, straight line between two points vs a less well fitting but predictive line across many points. It's a concept inherent to data rather than GPs or linear regression alone.

**Figure 3:** Differences between low- and high-data regimes. Note how whilst the forms of the curves are similar, the probabilities in the GP are vastly different.

<img src="assets/gp_data_update.png" width="720" alt="GP uncertainty with more data" />


### BO vs Design of Experiments (DoE)

- **DoE** uses fixed experimental designs based on mathematical criteria.
- **BO** adapts after every experiment, using data to update the model and choose the next best experiment.

This addresses one of the big limitations of DoE - DoE designs are a fixed size, and often sample data which is far away from your optimum which therefore provides little information towards reaching your goal


### Starting off

Lets explore how BO works. Our aim is to find the optimum, in this case the maximum, of a particular function.

For teaching purposes, we choose this function, `f(x)` to be a sine wave.

In real life, this function might be how your yield relates to temperature, time and reaction conditions. Usually, we do not know this function (hence we generate a model), but in this example case, we can take a look at our true function. In real experiments, each evaluation of `f` is costly.


In [ ]:
# --- Imports ---
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm


In [ ]:
def f(x: int | np.ndarray) -> int | np.ndarray:
    # Toy objective; in real life this is unknown and expensive to evaluate.
    return np.sin(3 * x) + 0.5 * x


# Candidate grid (for plotting and acquisition evaluation).
xs = np.linspace(0, 2, 400)
# True values (used only for plotting, not by the optimiser).
ys = f(xs)
plt.plot(xs, ys)
plt.title("True function (unknown to optimiser)")
plt.show()


## Surrogate model: Gaussian Process

A GP models a _distribution_ over functions. For any `x`, it gives a mean prediction and an uncertainty estimate, which lets BO decide where to sample next. In this case, we've said that before we let BO take over, we've sampled two initial points in `x`, defined in the array `X_train` as `x = 0.1 or 0.2`. Try editing this array to see how it changes what the GP thinks is happening.

The resulting GP acts as a surrogate model (a term frequently heard in ML literature); i.e. it is a surrogate of the actual function.

A Gaussian Process is built around a **kernel**, which describes how changes to the input affect the output. There are numerous kernels which model different degrees of "smoothness/roughness" of the output landscape, and the choice of kernel (and its associated hyperparameters) is one user-defined aspect which itself can be optimized in the application of BO.

In a chemistry context, the kernel could describe how changing reaction conditions is expected to change the result - for example, whether small tweaks to temperature give small or sudden jumps in yield, whether some patterns repeat, and how far the effect of one experiment “spreads” to nearby conditions.

Once you run the code below, you'll see what BO thinks the true function is based on the two points we sampled.


In [ ]:
# Initial experimental data (two observations).
X_train = np.array([[0.1], [0.2]])  # Change this to modify initial data.
y_train = f(X_train)

# Define a GP prior: smooth function + noise.
kernel = ConstantKernel(1.0) * Matern(length_scale=0.5, nu=2.5) + WhiteKernel()

# Fit GP surrogate to observed data.
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=1)
gp.fit(X_train, y_train)

# Predict mean/uncertainty over the candidate grid.
mu, std = gp.predict(xs.reshape(-1, 1), return_std=True)

plt.plot(xs, mu)
plt.fill_between(xs, mu - std, mu + std, alpha=0.3)
plt.scatter(X_train, y_train, c="red", s=40, label="Observed points")
plt.legend()


## Visualising uncertainty

Instead of a single prediction, a GP provides a range of plausible values. This can be visualised using percentile contours - think of each possible point on the black line as a Gaussian distribution.


In [ ]:
from scipy.stats import norm

# Predictive percentiles illustrate uncertainty at each x.
mu, std = gp.predict(xs.reshape(-1, 1), return_std=True)

percentiles = np.arange(5, 96, 10)  # 5%, 15%, ..., 95%
colors = plt.cm.Blues(np.linspace(0.3, 1.0, len(percentiles)))

plt.figure(figsize=(10, 6))

for p, c in zip(percentiles, colors):
    z = norm.ppf(p / 100)
    q = mu + z * std
    label = f"{p}%"
    plt.plot(xs, q, color=c, lw=1, label=label)

# Median line
plt.plot(xs, mu, color="black", lw=2, label="50% (median)")

# Observed data
plt.scatter(X_train, y_train, c="red", s=40, label="Observed points")

plt.title("GP Predictive Percentile Contours (5%?95% Every 10%)")
plt.xlabel("x")
plt.ylabel("Predicted y")
plt.legend()
plt.show()


## Acquisition Functions in Bayesian Optimisation

### What is an acquisition function?

Once the Gaussian Process (GP) has built a model of your reaction landscape, you still need to decide:

> **Which experiment should be run next?**

The **acquisition function** is the rule that combines

- the GP’s predicted mean (what looks good), and
- the GP’s uncertainty (what is still unknown),

into a single score for every possible set of conditions.

As such, the acquisition function helps us to find a good trade-off between exploitation and exploration.

- **Exploit** – go to conditions that already look good, or
- **Explore** – go to uncertain regions that might hide better chemistry.

Each acquisition function represents a different exploration strategy. We're using Expected Improvement, EI, but the choice of acquisition function is another user-defined component which contibutes to optimisation success.

### Acquisition function: Expected Improvement (EI)

EI is an acquisition function which scores more highly where the model predicts a high mean _and_/_or_ where uncertainty is large, so it naturally balances exploration vs exploitation. In a chemistry context, EI serves to answer the question:

> “Choose conditions where you might realistically beat your current best reaction.”

EI scores points based on how much improvement over the current best yield is expected.


### The optimisation loop in words

At each step:

1. Fit the GP to all observed data.
2. Compute EI over a grid of possible x values.
3. Pick the x with the highest EI as the next experiment.
4. Observe the new result and add it to the dataset.

In real chemistry, step 4 would be a lab experiment; here we call the toy function.


In [ ]:
def expected_improvement(x, gp, y_best, eps=1e-9):
    # Expected improvement for maximisation.
    # Predict mean and std from GP.
    mu, std = gp.predict(x.reshape(-1, 1), return_std=True)

    # Prevent division by zero.
    std = np.maximum(std, eps)

    z = (mu - y_best) / std
    ei = (mu - y_best) * norm.cdf(z) + std * norm.pdf(z)

    # EI should be zero at points with zero variance.
    ei[std < 1e-8] = 0.0

    return ei


## The BO loop

At each iteration we:

1. fit the GP to all observed data,
2. score candidates with EI,
3. pick the best candidate, and
4. observe the new result.


In [ ]:
X_train = np.array([[0.1], [0.2]])
y_train = f(X_train)

for i in range(10):
    # Fit GP to current data.
    gp.fit(X_train, y_train)

    # Predict current GP mean and uncertainty.
    mu, std = gp.predict(xs.reshape(-1, 1), return_std=True)

    # Compute EI for the acquisition step.
    ei = expected_improvement(xs, gp, np.max(y_train))

    # Choose next point on the grid (argmax of EI).
    x_next = xs[np.argmax(ei)]
    # In reality this would be a new experiment.
    y_next = f(x_next)

    # ---- UPDATE TRAINING DATA ----
    X_train = np.vstack([X_train, [x_next]])
    y_train = np.append(y_train, y_next)

    # ---- PLOT (this must use *new* mu/std/EI next iteration) ----
    plt.figure(figsize=(8, 5))

    plt.plot(xs, ys, "k--", label="True f")
    plt.plot(xs, mu, "b", label="GP mean")
    plt.fill_between(xs, mu - std, mu + std, alpha=0.2)

    plt.scatter(X_train, y_train, c="red", s=40)

    ei_scaled = ei / np.max(ei) * (ys.max() - ys.min()) + ys.min()
    plt.plot(xs, ei_scaled, "g", alpha=0.6, label="Scaled EI")

    plt.axvline(x_next, color="orange", linestyle="--")
    plt.title(f"Iteration {i}: Selected x = {x_next:.3f}")
    plt.legend()
    plt.show()

    print(f"Iteration {i}: x={x_next:.3f}, y={y_next:.3f}")


## Randomness and reproducibility

Bayesian optimisation often includes randomness in:

- the initial design (random starting experiments),
- the GP hyperparameter search (random restarts), and
- measurement noise in real experiments.

In this notebook we keep most steps deterministic: the initial points are fixed and EI is evaluated on a fixed grid. The only stochastic element is the GP hyperparameter optimization via `n_restarts_optimizer` (controlled by `random_state=1`). We also include a `WhiteKernel` so the GP can model noisy observations, even though the toy function is noise-free.


## Diagnostics

We track simple diagnostics to see whether optimisation is improving:

- best-so-far (monotonic improvement of the best observed value), and
- regret (distance from the true optimum; only available in this toy example).

In real campaigns you usually cannot compute regret, so you track improvement over a baseline, model uncertainty, or cross-validated error instead.


In [ ]:
# Best-so-far curve (monotonic improvement).
best_so_far = np.maximum.accumulate(y_train)

plt.figure(figsize=(7, 4))
plt.plot(best_so_far, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Best-so-far value")
plt.title("Convergence of optimisation")
plt.grid(True)
plt.show()

# Regret = gap to the true optimum (known only in this toy example).
opt_y = np.max(ys)
regret = opt_y - best_so_far

plt.figure(figsize=(7, 4))
plt.plot(regret, marker="o")
plt.yscale("log")  # typical for regret
plt.xlabel("Iteration")
plt.ylabel("Regret (distance from optimum)")
plt.title("Regret Curve")
plt.grid(True)
plt.show()

# Expected improvement (max EI each iteration).
ei_history = []

gp_diag = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=1)

for i in range(2, len(X_train)):
    gp_diag.fit(X_train[:i], y_train[:i])
    ei = expected_improvement(xs, gp_diag, np.max(y_train[:i]))
    ei_history.append(np.max(ei))

plt.figure(figsize=(7, 4))
plt.plot(ei_history, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Max expected improvement")
plt.title("Expected Improvement per iteration")
plt.grid(True)
plt.show()


## Knowing when you are done

As observed, the optimiser continues despite being very close to the true optimum. In reality, you stop BO when further experiments are not worth the cost. Common stopping criteria:

- You hit a fixed budget (time, materials, or number of runs).
- Best-so-far has plateaued for several iterations.
- The acquisition values are uniformly small (little expected improvement anywhere).
- Model uncertainty is low in the region of interest, or predicted gains are smaller than experimental noise.


## Exporting your data

Here we've provided a way for you to save experiment runs. It's a good idea to keep track of these if you want to interpret these runs at a different point in time, as making small modifications to the BO code mean that often runs are not easily reproducible.


In [ ]:
# Export the experiment history
import pandas as pd

# Fit GP on the final dataset for predictions
gp.fit(X_train, y_train)

x_vals = np.array(X_train).reshape(-1)
y_vals = np.array(y_train).reshape(-1)

# Model predictions at observed points
mu_obs, std_obs = gp.predict(np.array(X_train), return_std=True)

# Suggest the next experiment (EI) on the grid
mu_grid, std_grid = gp.predict(xs.reshape(-1, 1), return_std=True)
ei_grid = expected_improvement(xs, gp, np.max(y_vals))
x_next = xs[np.argmax(ei_grid)]
mu_next, std_next = gp.predict(np.array([[x_next]]), return_std=True)
ei_next = expected_improvement(np.array([x_next]), gp, np.max(y_vals))[0]

# Predicted best by GP mean (not necessarily the EI choice)
best_idx = np.argmax(mu_grid)
pred_best_x = xs[best_idx]
pred_best_mu = mu_grid[best_idx]

# EI at observed points
# PR: The EI is almost zero for all steps. This is because the observed points are all used to train the GP, and therefore have low uncertainty.
# It would be more informative to track EI during optimization.
ei_obs = expected_improvement(x_vals, gp, np.max(y_vals))

df = pd.DataFrame(
    {
        "experiment": np.arange(len(x_vals)),
        "x": x_vals,
        "y": y_vals,
        "mu_pred": mu_obs.reshape(-1),
        "std_pred": std_obs.reshape(-1),
        "best_so_far": np.maximum.accumulate(y_vals),
        "ei": ei_obs.reshape(-1),
    }
)

next_row = {
    "experiment": len(df),
    "x": float(x_next),
    "y": np.nan,
    "mu_pred": mu_next.item(),
    "std_pred": std_next.item(),
    "best_so_far": float(df["best_so_far"].iloc[-1]),
    "ei": float(ei_next),
}
df = pd.concat([df, pd.DataFrame([next_row])], ignore_index=True)

save_csv = False  # set True to write to disk
csv_path = "bo_experiments.csv"

if save_csv:
    df.to_csv(csv_path, index=False)
    print(f"Saved {len(df)} experiments to {csv_path}")

df
